# BBEATSx Phase 3: Decomposition Fidelity & Component UQ — DGP 2

This notebook validates BBEATSx on **DGP 2: Nonlinear Trend + Multi-Period Seasonality + AR(2) + Volatility Clustered (SV) Noise**.
It measures the forecasting calibration battery and the empirical coverage of component-level credible bands under stochastic volatility.

## Google Colab Setup

In [ ]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab. Setting up workspace...")
    import os
    if not os.path.abspath(".").endswith("simulations"):
        if not os.path.exists("BBEATSx"):
            !git clone https://github.com/hugogobato/BBEATSx.git
        !pip install ./BBEATSx
        %cd BBEATSx/simulations
    else:
        print("Already in simulations directory.")

import sys
import os
sys.path.append(os.path.abspath("."))
sys.path.append(os.path.abspath(".."))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bbeatsx import BBEATSx, make_config
import utils

## Simulation Configuration

In [ ]:
NUM_SIMULATIONS = 100  # Set to 100 for final results on Colab
HORIZON = 20
LEVEL = 0.90
print(f"Running {NUM_SIMULATIONS} trials for DGP 2...")

## Run Simulations

In [ ]:
dgp2_forecasts = []
dgp2_components = []

for i in range(NUM_SIMULATIONS):
    t, y, true_comps = utils.generate_dgp2(n=250, seed=i)
    n_train = len(y) - HORIZON
    y_train, y_test = y[:n_train], y[n_train:]
    
    cfg = make_config(
        periods=[(12, 3), (7, 2)],
        lags=(1, 2),
        trend="spline",
        errors="sv",
        num_gfr=10,
        num_burnin=150,
        num_mcmc=300,
        seed=i
    )
    
    model = BBEATSx(cfg).fit(y_train)
    fc = model.forecast(HORIZON)
    
    dgp2_forecasts.append(utils.evaluate_forecast(y_test, fc, y_train, level=LEVEL))
    
    s = model.sampler_
    mean, std = s.y_mean_, s.y_std_
    in_sample_draws = s.in_sample_components()
    pred_in_sample = {
        "trend": mean + std * in_sample_draws["trend"],
        "seasonal": std * in_sample_draws["seasonal"],
        "generic": std * in_sample_draws["generic"]
    }
    comp_metrics_in = utils.evaluate_component_fidelity(
        true_comps, pred_in_sample, offset=s.fs.row_offset, level=LEVEL
    )
    
    pred_out_sample = {
        "trend": fc.components["trend"],
        "seasonal": fc.components["seasonal"],
        "generic": fc.components["generic"]
    }
    true_out_comps = {k: v[n_train:] for k, v in true_comps.items()}
    comp_metrics_out = utils.evaluate_component_fidelity(
        true_out_comps, pred_out_sample, offset=0, level=LEVEL
    )
    
    dgp2_components.append({
        "in_sample": comp_metrics_in,
        "out_sample": comp_metrics_out
    })
    
    if (i + 1) % 10 == 0:
        print(f"  Completed trial {i + 1}/{NUM_SIMULATIONS}")

## Results Summary

In [ ]:
df_fc2 = pd.DataFrame(dgp2_forecasts)
print("\n--- DGP 2 Forecasting Calibration Battery ---")
print(df_fc2.mean().to_frame("Mean Metric"))

comp_in_rmse = {k: np.mean([t["in_sample"][k]["rmse"] for t in dgp2_components]) for k in ["trend", "seasonal", "generic"]}
comp_in_cov = {k: np.mean([t["in_sample"][k]["coverage"] for t in dgp2_components]) for k in ["trend", "seasonal", "generic"]}
comp_out_rmse = {k: np.mean([t["out_sample"][k]["rmse"] for t in dgp2_components]) for k in ["trend", "seasonal", "generic"]}
comp_out_cov = {k: np.mean([t["out_sample"][k]["coverage"] for t in dgp2_components]) for k in ["trend", "seasonal", "generic"]}

df_comp2 = pd.DataFrame({
    "In-Sample RMSE": comp_in_rmse,
    "In-Sample Coverage (90%)": comp_in_cov,
    "Out-of-Sample RMSE": comp_out_rmse,
    "Out-of-Sample Coverage (90%)": comp_out_cov
})
print("\n--- DGP 2 Component Fidelity (Plan §3.2) ---")
print(df_comp2)

## Visualizations

In [ ]:
from bbeatsx.interpret import plot_decomposition

fig, ax = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
plot_decomposition(model.sampler_, level=LEVEL, ax=ax)
ax[0].set_title("BBEATSx Component Decomposition with 90% Credible Bands (DGP 2)")
plt.tight_layout()
plt.show()